# F1 · Inventario del ground truth

Enumera las unidades `(serie, etiqueta)`, fija el `start` de cada una y mide cada par
`(unidad, frame)`. La condicion de salida de la fase es que salgan **17 unidades** y
**163 frames anotados**, y que el reparto de los **312 pares** cuadre serie a serie con
lo que declara el config.

Este notebook no contiene logica: llama a `surco.inventario` y muestra.

### Procedencia

**Qué hay que haber ejecutado antes:** nada más que tener `data/` en su sitio. Este notebook
es el primer eslabón de la cadena y **calcula desde las imágenes y el GT**, no desde ningún
CSV.

```bash
conda env create -f environment.yml
conda run -n surco pip install -e .
```

**Qué produce**, en `salidas/proceso/`: `inventario_gt.csv` (una fila por par unidad-frame) y
`unidades.csv` (una por unidad, con su `start`). De los dos comen F2, F3, el estudio de
concordancia y el cruce del desacuerdo humano con la fragmentación. **Corre entero en el
entorno `surco`**, sin GPU y sin ningún modelo.

`fragmentacion_en_start.csv` sale de aquí en el sentido de que se calcula sobre este
inventario, pero **lo escribe F2**, que es donde se mira: dice qué unidades ya llegan
partidas al frame sobre el que se anotan los clics.

El mapa completo del proyecto está en `GUIA.ipynb`.


In [1]:
import pandas as pd

from surco import config, datos, inventario

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 60)

## Las series se leen

Cada `.TIF` en disco es el mini Z-stack de 5 planos; lo que se usa es el maximo.
Tarda unos 20 s porque son 2 GB de lectura.

In [2]:
n_imagenes = datos.comprobar_imagenes()
frame = datos.cargar_frame(5, 3)
print(f"{n_imagenes} imagenes leidas; una de ellas: {frame.shape} {frame.dtype}")

210 imagenes leidas; una de ellas: (1024, 1024) uint16


## Las unidades

Una unidad es la pareja `(serie, etiqueta)`. Las etiquetas de una serie salen de unir
los valores distintos de fondo de sus 21 frames, no de leer una sola mascara: el numero
de heridas cambia dentro de la serie. El `start` es por unidad, el primer frame en que
esa herida aparece.

In [3]:
inv = inventario.construir_inventario()
unidades = inventario.tabla_unidades(inv)
unidades

,serie,etiqueta,start,ultimo,n_pares,contiguo,salto_centroide_max_px,centroide_col_media
0,1,1,3,21,19,True,17.835860,235.882785
1,1,2,4,21,18,True,8.531394,327.620281
2,2,1,4,21,18,True,4.925025,457.936024
3,2,2,4,21,18,True,11.313507,732.148820
4,3,1,4,21,18,True,12.397487,522.505729
5,4,1,3,21,19,True,13.188651,462.300897
6,4,2,3,21,19,True,8.440612,517.716241
7,5,1,3,21,19,True,7.865293,111.868177
8,5,2,3,21,19,True,13.882072,503.553006
9,5,3,4,21,18,True,11.314134,818.203437


## Los recuentos contra el config

In [4]:
anotados = inventario.recuento_frames_anotados()

resumen = pd.DataFrame(
    [
        ("unidades", len(unidades), config.N_UNIDADES),
        ("pares", len(inv), config.N_PARES),
        ("frames anotados", int(anotados["n_frames_anotados"].sum()), config.N_FRAMES_ANOTADOS),
    ],
    columns=["magnitud", "medido", "config"],
)
resumen["cuadra"] = resumen["medido"] == resumen["config"]
resumen

,magnitud,medido,config,cuadra
0,unidades,17,17,True
1,pares,312,312,True
2,frames anotados,163,163,True


In [5]:
por_serie = inv.groupby("serie").size().rename("medido").to_frame()
por_serie["config"] = pd.Series(config.PARES_POR_SERIE)
por_serie["cuadra"] = por_serie["medido"] == por_serie["config"]
por_serie

,medido,config,cuadra
serie,,,
1,37,37,True
2,36,36,True
3,18,18,True
4,38,38,True
5,56,56,True
6,13,13,True
7,38,38,True
9,38,38,True
10,38,38,True


## Estabilidad de las etiquetas

La regla del GT es que las heridas se numeran de izquierda a derecha. Lo que la romperia
es que apareciera una nueva **a la izquierda** de las existentes: se llevaria el 1 y
desplazaria a las demas. Se comprueba sin umbral, por el orden de las columnas y por si
alguna unidad tardia cae a la izquierda de otra anterior. El salto de centroide se
reporta como diagnostico, no como criterio.

In [6]:
inventario.orden_de_etiquetas(unidades)

,serie,n_unidades,etiquetas_ordenadas_por_columna,tardia_a_la_izquierda,salto_centroide_max_px
0,1,2,True,False,17.835860
1,2,2,True,False,11.313507
2,3,1,True,False,12.397487
3,4,2,True,False,13.188651
4,5,3,True,False,13.882072
5,6,1,True,False,7.954833
6,7,2,True,False,12.460474
7,9,2,True,False,11.820770
8,10,2,True,False,9.096373


## Fragmentacion

El numero de componentes es propiedad del par `(unidad, frame)`, no un booleano por
lesion. La conectividad canonica es la de **8 vecinos**, `config.COLUMNA_FRAGMENTACION`;
`n_comp_c4` se conserva solo como diagnostico y se muestra al lado.

In [7]:
fragmentacion = pd.DataFrame(
    {
        "pares_partidos": [(inv[c] > 1).sum() for c in ("n_comp_c4", "n_comp_c8")],
        "fraccion": [(inv[c] > 1).mean() for c in ("n_comp_c4", "n_comp_c8")],
        "max_componentes": [inv[c].max() for c in ("n_comp_c4", "n_comp_c8")],
    },
    index=["4 vecinos", "8 vecinos"],
)
fragmentacion

,pares_partidos,fraccion,max_componentes
4 vecinos,101,0.323718,7
8 vecinos,99,0.317308,7


In [8]:
inv.groupby("frame")[config.COLUMNA_FRAGMENTACION].mean().rename(
    "componentes medias por frame"
)

frame
3     1.090909
4     1.062500
5     1.000000
6     1.062500
7     1.375000
8     1.250000
9     1.411765
10    1.352941
11    1.647059
12    1.705882
13    1.764706
14    2.058824
15    2.000000
16    2.000000
17    2.000000
18    2.000000
19    2.058824
20    2.117647
21    2.117647
Name: componentes medias por frame, dtype: float64

## Mascaras duplicadas

Frames consecutivos cuya ROI es identica pixel a pixel: se copio en vez de redibujarse.
No son anotaciones independientes y sobran en la muestra de concordancia del apartado 7.

In [9]:
inventario.duplicados_de_todas_las_series()

,serie,etiqueta,frame_a,frame_b
0,1,1,13,14
1,1,1,18,19
2,1,2,11,12
3,4,1,20,21
4,4,2,7,8
5,4,2,15,16
6,5,2,8,9
7,5,2,18,19
8,10,1,10,11
9,10,1,12,13


## Geometria medida contra la estimada

Los valores del config son estimaciones del trabajo anterior, obtenidas modelando el
surco como una elipse. Aqui se miden. No hay `SEMIANCHO_PX` global: F2 usa el de cada
unidad.

In [10]:
geometria = pd.DataFrame(
    [
        ("area_px2", inv["area_px2"].mean(), config.AREA_MEDIA_PX2),
        ("elongacion", inv["elongacion"].mean(), config.ELONGACION_GT),
        ("semiancho_px", inv["semiancho_px"].mean(), config.SEMIEJE_MENOR_PX),
    ],
    columns=["magnitud", "medido", "estimado en el config"],
)
geometria

,magnitud,medido,estimado en el config
0,area_px2,1653.990385,1552.00
1,elongacion,7.396193,6.91
2,semiancho_px,10.699913,8.50


In [11]:
semiancho = inv.groupby(["serie", "etiqueta"])["semiancho_px"].agg(["mean", "min", "max"])
semiancho

mean        min        max
serie etiqueta                                 
1     1          8.352738   5.582885  14.183664
      2          5.495203   4.914875   6.865386
2     1          9.628699   8.569379  10.767271
      2          6.817736   5.837760   8.515267
3     1          7.832606   5.857445  10.623972
4     1          8.271917   6.819539   9.064411
      2          7.951763   7.257561   8.776908
5     1         11.211227   9.517507  12.809886
      2         10.543163   8.624925  13.537786
      3          8.873421   6.560867  11.204058
6     1         11.245940   7.520738  14.194391
7     1         21.275137  13.628919  26.798962
      2         16.218907  10.092210  19.520252
9     1          9.474582   8.158974  11.115108
      2         15.172885  12.576853  17.588962
10    1         13.379010  12.229086  14.352184
      2          9.544330   6.984438  11.894340

## Salidas

`inventario_gt.csv` es una fila por par `(unidad, frame)` y es la entrada de F2.

In [12]:
inv.to_csv(config.DIR_PROCESO / "inventario_gt.csv", index=False)
unidades.to_csv(config.DIR_PROCESO / "unidades.csv", index=False)
print(f"escritos en {config.DIR_PROCESO}")

escritos en E:\ws_Metricas\salidas\proceso
